# TP 02 : Analyse de Variance Hiérarchisée (Nested ANOVA)
**Master 2 Biochimie Appliquée — Université M'Hamed Bougara de Boumerdès (UMBB)**  
*Enseignante : Dr. Sarra BENMOUMOU (Ph.D.)*

---

## Contexte :
Contrôle qualité dans un procédé de bioproduction d'insuline recombinante.
- Facteur principal A : 3 Lots industriels indépendants ($I=3$).
- Facteur emboîté B : 3 Flacons par lot ($J=3$, $B \subset A$).
- Réplicats techniques : 3 mesures de pureté par flacon ($K=3$, $N=27$).

## Objectifs :
1. Calculer l'ANOVA hiérarchique et identifier le bon dénominateur pour le test $F_A$.
2. Évaluer si la variabilité provient des lots ou des flacons.


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df_nest = pd.read_csv('../datasets/production_insuline_nested.csv')
display(df_nest.head())


In [ ]:
# Calcul ANOVA Hiérarchisée
I = df_nest['Lot'].nunique()
J = 3 # 3 flacons par lot
K = 3 # 3 réplicats par flacon
N = len(df_nest)

grand_mean = df_nest['Purete_Pourcent'].mean()
SS_total = ((df_nest['Purete_Pourcent'] - grand_mean)**2).sum()

means_lot = df_nest.groupby('Lot')['Purete_Pourcent'].mean()
SS_lot = (J * K) * ((means_lot - grand_mean)**2).sum()

means_flacon = df_nest.groupby(['Lot', 'Flacon'])['Purete_Pourcent'].mean()
SS_flacon_total = K * ((means_flacon - grand_mean)**2).sum()
SS_flacon_dans_lot = SS_flacon_total - SS_lot

SS_res = SS_total - SS_flacon_total

df_lot = I - 1
df_flacon_dans_lot = I * (J - 1)
df_res = I * J * (K - 1)

MS_lot = SS_lot / df_lot
MS_flacon_dans_lot = SS_flacon_dans_lot / df_flacon_dans_lot
MS_res = SS_res / df_res

# IMPORTANT : Le test du facteur principal Lot est divisé par MS du facteur emboîté !
F_lot = MS_lot / MS_flacon_dans_lot
F_flacon = MS_flacon_dans_lot / MS_res

p_lot = 1 - stats.f.cdf(F_lot, df_lot, df_flacon_dans_lot)
p_flacon = 1 - stats.f.cdf(F_flacon, df_flacon_dans_lot, df_res)

table_nest = pd.DataFrame({
    'Source': ['Lots (A)', 'Flacons dans Lots B(A)', 'Erreur Technique (Rés)'],
    'ddl': [df_lot, df_flacon_dans_lot, df_res],
    'SS': [round(SS_lot, 2), round(SS_flacon_dans_lot, 2), round(SS_res, 2)],
    'MS': [round(MS_lot, 2), round(MS_flacon_dans_lot, 2), round(MS_res, 2)],
    'F_obs': [round(F_lot, 2), round(F_flacon, 2), np.nan],
    'p_value': [f'{p_lot:.4f}', f'{p_flacon:.4e}', np.nan]
})

print("=== TABLE ANOVA HIERARCHISEE ===")
display(table_nest)


## Interprétation Biologique & Décision Industrielle :
Le procédé de fermentation est-il stable ? D'où provient la source d'hétérogénéité constatée ?
